# 🛰️ Magnine (MANGANAI) — Run on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/funcodes-Sky7/Magnine/blob/main/colab_run.ipynb)

This notebook lets you run the full **Magnine AI Manganese Exploration Platform** (FastAPI backend + React frontend + live ML prospectivity map) directly on Google Colab's cloud environment.

### How to run:
1. Click **Runtime** → **Run all** (or press `Ctrl+F9` / `Cmd+F9`).
2. At the final cell, copy your tunnel password and click the provided web link to open the app!

### Step 1: Clone Repository & Install Node.js

In [ ]:
# 1. Clone repository
import os
if not os.path.exists('/content/Magnine'):
    !git clone https://github.com/funcodes-Sky7/Magnine.git /content/Magnine
%cd /content/Magnine

# 2. Install Node.js 20 and localtunnel
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!npm install -g localtunnel

### Step 2: Install Backend Dependencies

In [ ]:
%cd /content/Magnine/backend
!pip install -r requirements.txt

### Step 3: Install Frontend Dependencies & Build Production Bundle

In [ ]:
%cd /content/Magnine/frontend
# Configure frontend to speak to backend on localhost:8000
with open('.env', 'w') as f:
    f.write('VITE_API_BASE_URL=http://localhost:8000\nVITE_DEMO_MODE=true\n')

!npm install
!npm run build

### Step 4: Launch Backend & Frontend Services in Background

In [ ]:
import subprocess
import time

# 1. Start FastAPI backend (port 8000)
backend_process = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/Magnine/backend"
)

# 2. Start Vite frontend preview server (port 5173)
frontend_process = subprocess.Popen(
    ["npx", "vite", "preview", "--host", "0.0.0.0", "--port", "5173"],
    cwd="/content/Magnine/frontend"
)

time.sleep(4)
print("✅ Backend API running on port 8000")
print("✅ Frontend running on port 5173")

### Step 5: Open the Web Application (Localtunnel)

In [ ]:
import urllib.request

# Get tunnel password (Colab machine IP)
try:
    tunnel_pwd = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode('utf8').strip()
    print("===========================================================")
    print(f"🔑 YOUR TUNNEL PASSWORD IS: {tunnel_pwd}")
    print("===========================================================")
    print("1. Click the URL generated below.")
    print("2. Paste the password above into the 'Tunnel Password' field.")
    print("3. Click 'Click to Submit' to open Magnine!")
    print("===========================================================\n")
except Exception as e:
    print(f"Could not retrieve tunnel password automatically: {e}")

# Start tunnel on port 5173
!npx localtunnel --port 5173

--- 
### (Optional) Step 6: Retrain ML Model with Google Earth Engine
Run this cell only if you want to sample fresh Sentinel-2 + SRTM features and retrain the Random Forest model using your Google account:

In [ ]:
# Uncomment and run if you want to retrain the ML model:
# import ee
# ee.Authenticate()
# ee.Initialize(project='your-gcp-project-id')
# %cd /content/Magnine/backend
# !python -m app.ml.ml_trainer
# !python -m app.ml.run_inference